# Setup
## Necessary library imports

In [13]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from tabulate import tabulate
import numpy as np
import sqlite3
import re
import pandas as pd

# Extract 
## Getting the players, player_stats, teams and plays data from those CSV files

In [14]:
#Get players
file_path = "base_data/players.csv"

players_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get players stats
file_path = "playerStats_data/playerStats_2024_ENG.1.csv"

player_stats_2024_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get teams
file_path = "base_data/teams.csv"

teams_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

#Get plays
file_path = "plays_data/plays_2024_ENG.1.csv"

plays_df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "excel4soccer/espn-soccer-data",
  file_path,
)

/workspaces/Helix-Football-Data-App/myenvironment/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (0: nickName) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


# Transform
## Creating a unified dataframe with combined stats, calculated metrics, and percentiles

In [15]:
# Drop columns I won't be using in each df

players_df = players_df[[
    "athleteId",
    "fullName",
    "slug",
    "weight",
    "displayHeight",
    "height",
    "age",
    "dateOfBirth",
    "citizenship",
    "positionAbbreviation"
]]

player_stats_2024_df = player_stats_2024_df[[
    "teamId",
    "athleteId",
    "appearances_value",
    "foulsCommitted_value",
    "foulsSuffered_value",
    "yellowCards_value",
    "redCards_value",
    "ownGoals_value",
    "goalAssists_value",
    "offsides_value",
    "shotsOnTarget_value",
    "totalShots_value",
    "totalGoals_value",
    "shotsFaced_value",
    "saves_value",
    "goalsConceded_value"
]]

# Handling players who played for multiple clubs and appear as duplicates by aggregating their stats

aggregate_dict = {
    "teamId": "first", 
    "appearances_value": "sum",
    "foulsCommitted_value": "sum",
    "foulsSuffered_value": "sum",
    "yellowCards_value": "sum",
    "redCards_value": "sum",
    "ownGoals_value": "sum",
    "goalAssists_value": "sum",
    "offsides_value": "sum",
    "shotsOnTarget_value": "sum",
    "totalShots_value": "sum",
    "totalGoals_value": "sum",
    "shotsFaced_value": "sum",
    "saves_value": "sum",
    "goalsConceded_value": "sum"
}

player_stats_2024_df = (
    player_stats_2024_df
    .sort_values("appearances_value", ascending=False)
    .groupby("athleteId", as_index=False)
    .agg(aggregate_dict)
)

# Getting English first division team IDs so that I can isolate them in the teams dataframe
prem_teams_ids = player_stats_2024_df["teamId"].unique().tolist()

# Creating a dataframe tht's only English first division teams
prem_teams_df = teams_df[teams_df["teamId"].isin(prem_teams_ids)]

# Merge players.csv and player_stats.csv on athleteId
first_merged_df = players_df.merge(player_stats_2024_df, on="athleteId")

# Merge unified players dataframes with teams dataframe so that team name, colours and logos are added as columns
second_merged_df = first_merged_df.merge(
    teams_df[["teamId", "shortDisplayName", "color", "alternateColor", "logoURL"]]
    .drop_duplicates(subset=["teamId"]),
    on="teamId", how="left"
)

# Rename columns to something clearer
second_merged_df = second_merged_df.rename(columns={
    "shortDisplayName":"teamName",
    "color": "teamPrimaryColor",
    "alternateColor": "teamSecondaryColor",
    "logoURL": "teamLogo"})

# Parsing plays data to count chances created
def count_chances_created(plays_df):

    shot_types = [106, 117, 135, 70, 137, 173]

    plays_df = plays_df.drop_duplicates(subset=['eventId', "text"])

    assisted_shots_df = plays_df[
        plays_df["typeId"].isin(shot_types) &
        plays_df["text"].str.contains("Assisted by", na = False)
    ].copy()

    assisted_shots_df["assister_name"] = assisted_shots_df["text"].str.extract(
        r"Assisted by (.+?)(?:\.| with| following| from| after)"
    ).iloc[:, 0].str.strip()

    return (
        assisted_shots_df["assister_name"]
        .value_counts()
        .rename_axis("fullName")
        .reset_index()
        .rename(columns={"count": "chances_created"})
    )


chances_created_df = count_chances_created(plays_df)

In [16]:
# Drop columns I won't be using in each df

players_df = players_df[[
    "athleteId",
    "fullName",
    "slug",
    "weight",
    "displayHeight",
    "height",
    "age",
    "dateOfBirth",
    "citizenship",
    "positionAbbreviation"
]]

player_stats_2024_df = player_stats_2024_df[[
    "teamId",
    "athleteId",
    "appearances_value",
    "foulsCommitted_value",
    "foulsSuffered_value",
    "yellowCards_value",
    "redCards_value",
    "ownGoals_value",
    "goalAssists_value",
    "offsides_value",
    "shotsOnTarget_value",
    "totalShots_value",
    "totalGoals_value",
    "shotsFaced_value",
    "saves_value",
    "goalsConceded_value"
]]

# Getting English first division team IDs so that I can isolate them in the teams dataframe
prem_teams_ids = player_stats_2024_df["teamId"].unique().tolist()

# Creating a dataframe tht's only English first division teams
prem_teams_df = teams_df[teams_df["teamId"].isin(prem_teams_ids)]

# Merge players.csv and player_stats.csv on athleteId
first_merged_df = players_df.merge(player_stats_2024_df, on="athleteId")

# Merge unified players dataframes with teams dataframe so that team name, colours and logos are added as columns
second_merged_df = first_merged_df.merge(teams_df[["teamId", "shortDisplayName", "color","alternateColor", "logoURL"]], on="teamId", how="left")

# Rename columns to something clearer
second_merged_df = second_merged_df.rename(columns={
    "shortDisplayName":"teamName",
    "color": "teamPrimaryColor",
    "alternateColor": "teamSecondaryColor",
    "logoURL": "teamLogo"})

# Parsing plays data to count chances created
def count_chances_created(plays_df):

    shot_types = [106, 117, 135, 70, 137, 173]
    
    plays_df = plays_df.drop_duplicates(subset=['eventId', "text"])
    
    assisted_shots_df = plays_df[
        plays_df["typeId"].isin(shot_types) &
        plays_df["text"].str.contains("Assisted by", na = False)
    ].copy()

    assisted_shots_df["assister_name"] = assisted_shots_df["text"].str.extract(
        r"Assisted by (.+?)(?:\.| with| following| from| after)"
    ).iloc[:, 0].str.strip()

    return (
        assisted_shots_df["assister_name"]
        .value_counts()
        .rename_axis("fullName")
        .reset_index()
        .rename(columns={"count": "chances_created"})
    )          

chances_created_df = count_chances_created(plays_df)

In [17]:
# Adding calculated stats columns using numpy

unified_df = second_merged_df.copy()

# shot accuracy
unified_df["shot_accuracy"] = np.where (
    unified_df["totalShots_value"] > 0,
    np.round(
        unified_df["shotsOnTarget_value"] / unified_df["totalShots_value"] * 100,
        1
    ),
    np.nan
)

# conversion rate
unified_df["conversion_rate"] = np.where (
    unified_df["totalShots_value"] > 0,
    np.round(
        unified_df["totalGoals_value"] / unified_df["totalShots_value"] * 100,
        1
    ),
    np.nan
)

# on target conversion rate
unified_df["on_target_conversion_rate"] = np.where (
    unified_df["shotsOnTarget_value"] > 0,
    unified_df["totalGoals_value"] / unified_df["shotsOnTarget_value"],
    np.nan
)

# save percentage
unified_df["save_percentage"] = np.where (
    (unified_df["saves_value"] + unified_df["goalsConceded_value"] > 0) & (unified_df["positionAbbreviation"]=="G"),
    np.round(
        unified_df["saves_value"] / (unified_df["saves_value"]+ unified_df["goalsConceded_value"]) * 100,
        1
    ),
    np.nan
)

# discipline score
unified_df["discipline_score"] = unified_df["yellowCards_value"] + (unified_df["redCards_value"] * 2)

In [18]:
# Percentile calculation

# Percentile compared to players of the same position
def add_percentile_by_position(dataframe, column_name, new_column):
    for position in ["M","F","D", "G"]:
        position_match = (dataframe["positionAbbreviation"] == position)
        # choosing rows where the position is matched and grabbing column name
        # then ranking them based on percentile        
        dataframe.loc[position_match, new_column] = dataframe.loc[position_match, column_name].rank(pct=True) * 100
    return dataframe

# Percentile compared to every player
def add_percentile_global(dataframe, column_name, new_column):
    dataframe[new_column] = dataframe[column_name].rank(pct=True) * 100
    return dataframe

In [19]:
# Testing positional

test_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_percentile")
forwards = test_df[test_df["positionAbbreviation"] == "F"]
print(forwards[['fullName', 'totalGoals_value', 'goals_percentile']].sort_values('goals_percentile', ascending=False).head(10))
print()

# Testing global
test_df = add_percentile_global(test_df, "totalGoals_value", "goals_percentile")
print(test_df[["fullName", "positionAbbreviation", "totalGoals_value", "goals_percentile"]].sort_values("goals_percentile", ascending=False).head(10))


                 fullName  totalGoals_value  goals_percentile
132         Mohamed Salah                29        100.000000
297        Alexander Isak                23         99.462366
359        Erling Haaland                22         98.924731
411          Bryan Mbeumo                20         98.118280
51             Chris Wood                20         98.118280
250           Yoane Wissa                19         97.311828
187         Ollie Watkins                16         96.774194
378         Matheus Cunha                15         96.236559
317  Jean-Philippe Mateta                14         95.430108
357  Jørgen Strand Larsen                14         95.430108

                 fullName positionAbbreviation  totalGoals_value  \
132         Mohamed Salah                    F                29   
297        Alexander Isak                    F                23   
359        Erling Haaland                    F                22   
411          Bryan Mbeumo                    

In [ ]:
# Creating position specific percentiles for outfield players (for my radar charts)

unified_df = add_percentile_by_position(unified_df, "totalGoals_value", "goals_pct_pos")
unified_df = add_percentile_by_position(unified_df, "goalAssists_value", "assists_pct_pos")
unified_df = add_percentile_by_position(unified_df, "shot_accuracy", "shot_accuracy_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsSuffered_value", "fouls_suff_pct_pos")
unified_df = add_percentile_by_position(unified_df, "foulsCommitted_value", "fouls_comm_pct_pos")
unified_df= add_percentile_by_position(unified_df, "chances_created", "chances_created_pct_pos")

# For forwards
unified_df = add_percentile_by_position(unified_df, 'conversion_rate', 'conversion_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'on_target_conversion_rate', 'on_target_conv_pct_pos')

# Creating global percentiles for leaderboards
unified_df = add_percentile_global(unified_df, "totalGoals_value", "goals_pct_global")
unified_df = add_percentile_global(unified_df, "goalAssists_value", "assists_pct_global")
unified_df = add_percentile_global(unified_df, "shot_accuracy", "shot_accuracy_pct_global")
unified_df = add_percentile_global(unified_df, "foulsSuffered_value", "fouls_suff_pct_global")
unified_df = add_percentile_global(unified_df, "foulsCommitted_value", "fouls_comm_pct_global")

# Inverted metrics

# GK metrics
unified_df = add_percentile_by_position(unified_df, 'goalsConceded_value', 'goals_conc_pct_pos')
unified_df['goals_conc_pct_pos'] = 100 - unified_df['goals_conc_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'save_percentage', 'save_pct_pos')
unified_df = add_percentile_by_position(unified_df, 'saves_value', 'saves_pct_pos')

# Discipline metrics (defenders)
unified_df = add_percentile_by_position(unified_df, 'yellowCards_value', 'yellow_cards_pct_pos')
unified_df['yellow_cards_pct_pos'] = 100 - unified_df['yellow_cards_pct_pos']

unified_df = add_percentile_by_position(unified_df, 'redCards_value', 'red_cards_pct_pos')
unified_df['red_cards_pct_pos'] = 100 - unified_df['red_cards_pct_pos']

# Offsides (forwards)
unified_df = add_percentile_by_position(unified_df, 'offsides_value', 'offsides_pct_pos')
unified_df['offsides_pct_pos'] = 100 - unified_df['offsides_pct_pos']

In [21]:
# Merging chances created df with unified df 

# Removing duplicates caused by 
name_to_id = unified_df[["athleteId", "fullName"]].drop_duplicates()

# Deduplicate chances data
chances_created_df = chances_created_df.drop_duplicates(subset=["fullName"])

# Correcting name mismatches between plays text and players_df
name_corrections = {
    "Amad Diallo": "Amad",
    "Tino Livramento": "Valentino Livramento",
    "Jesper Lindstrøm": "Jesper Lindstrom",
    "Pape Sarr": "Pape Matar Sarr",
    "Will Smallbone": "William Smallbone",
    "Ederson": "Ederson ",
    "Sam Szmodics": "Sammie Szmodics",
    "Ben Brereton": "Ben Brereton Díaz",
    "Josh King": "Joshua King",
    "Ali Al-Hamadi": "Ali Ibrahim Al-Hamadi",
    "Antony": "Antony ",
    "Kepa": "Kepa Arrizabalaga",
    "Albert Grønbæk": "Albert Grønbaek",

    "Andy Robertson": "Andrew Robertson",
    "Vitalii Mykolenko": "Vitaliy Mykolenko",
    "Maximilian Kilman": "Max Kilman",
    "Abdul Fatawu": "Fatawu Issahaku",
    "Roméo Lavia": "Romeo Lavia",
    "Jeffrey Schlupp": "Jeff Schlupp",
    "Antonín Kinsky": "Antonin Kinsky",

    "Ollie Scarles": "Oliver Scarles",
    "Sam Amo-Ameyaw": "Samuel Amo-Ameyaw"
}

chances_created_df["fullName"] = chances_created_df["fullName"].replace(name_corrections)

# Adding IDs 
chances_with_ids = chances_created_df.merge(
    name_to_id,
    on="fullName",
    how="left"
)

# Checking unmatched records
unmatched = chances_with_ids[chances_with_ids["athleteId"].isna()]["fullName"].tolist()
if unmatched:
    print(f"Unmatched ({len(unmatched)}):", unmatched)

# Preventing merge error by dropping existing chances_created 
if "chances_created" in unified_df.columns:
    unified_df = unified_df.drop(columns=["chances_created"])

# Merge chances with unified, removing bad rows
unified_df = unified_df.merge(
    chances_with_ids[["athleteId", "chances_created"]].dropna(subset=["athleteId"]),
    on="athleteId",
    how="left"
)

# Clean missing values by replacing with 0
unified_df["chances_created"] = unified_df["chances_created"].fillna(0).astype(int)

Unmatched (2): ['Thiago', 'Harry Clarke']


In [22]:
# Putting colours in proper hex format
unified_df["teamPrimaryColor"] = "#" + unified_df["teamPrimaryColor"]
unified_df["teamSecondaryColor"] = "#" + unified_df["teamSecondaryColor"]

# Load
## Storing the unified dataframe as a CSV file to be accessed later

In [23]:
print(unified_df.columns)

Index(['athleteId', 'fullName', 'slug', 'weight', 'displayHeight', 'height',
       'age', 'dateOfBirth', 'citizenship', 'positionAbbreviation', 'teamId',
       'appearances_value', 'foulsCommitted_value', 'foulsSuffered_value',
       'yellowCards_value', 'redCards_value', 'ownGoals_value',
       'goalAssists_value', 'offsides_value', 'shotsOnTarget_value',
       'totalShots_value', 'totalGoals_value', 'shotsFaced_value',
       'saves_value', 'goalsConceded_value', 'teamName', 'teamPrimaryColor',
       'teamSecondaryColor', 'teamLogo', 'shot_accuracy', 'conversion_rate',
       'on_target_conversion_rate', 'save_percentage', 'discipline_score',
       'goals_percentile', 'goals_pct_pos', 'assists_pct_pos',
       'shot_accuracy_pct_pos', 'fouls_suff_pct_pos', 'fouls_comm_pct_pos',
       'conversion_pct_pos', 'on_target_conv_pct_pos', 'goals_pct_global',
       'assists_pct_global', 'shot_accuracy_pct_global',
       'fouls_suff_pct_global', 'fouls_comm_pct_global', 'goals_conc_p

In [24]:
# Saving the unified df as a CSV

unified_df.to_csv('../data/processed/unified_players_ENG_1_2024.csv', index=False)

In [25]:
# Saving it to SQLite for more efficient querying

myconnection = sqlite3.connect('../data/processed/helix_football_data.db')
# Saving my dataframe to a table called players in that database
unified_df.to_sql("players", myconnection, if_exists="replace", index=False)
# Closing the connection
myconnection.close()

print(f"Saved {len(unified_df)} players to helix_football_data.db")

Saved 766 players to helix_football_data.db
